# Dataset : CIFAR-10  
            60,000 colour images (32×32×3) across 10 object classes:
            airplane, automobile, bird, cat, deer,
            dog, frog, horse, ship, truck
  Libraries: NumPy · Pandas · Matplotlib · Seaborn · Scikit-Learn · TensorFlow

In [ ]:
# Imports
import os, time, warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

import seaborn as sns
warnings.filterwarnings("ignore")


In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.decomposition   import PCA
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline        import Pipeline

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Output directory
OUT = "cifar10_outputs"
os.makedirs(OUT, exist_ok=True)

CLASS_NAMES = ["airplane","automobile","bird","cat","deer",
               "dog","frog","horse","ship","truck"]

In [ ]:
# Colour palette
BG_DARK  = "#0d0d1a"
BG_PANEL = "#16213e"
BG_MID   = "#1a1a2e"
ACCENT   = "#4C72B0"
GREEN    = "#2ecc71"
RED      = "#e74c3c"

def _style(fig, *axes):
    fig.patch.set_facecolor(BG_DARK)
    for ax in axes:
        ax.set_facecolor(BG_PANEL)
        ax.tick_params(colors="white")
        for sp in ax.spines.values():
            sp.set_edgecolor("#444466")

print("="*72)
print("  ML JOURNEY — CIFAR-10 Image Classification")
print(f"  TensorFlow {tf.__version__}  |  NumPy {np.__version__}  |  Pandas {pd.__version__}")
print("="*72)

results = {}   # name → {accuracy, train_time, history, type}




In [ ]:
#  DATA LOADING & EXPLORATORY DATA ANALYSIS
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.cifar10.load_data()
y_tr_raw = y_tr_raw.flatten()
y_te_raw = y_te_raw.flatten()

print(f"  Train : {X_tr_raw.shape}  |  Test : {X_te_raw.shape}")
print(f"  Pixel range : [{X_tr_raw.min()}, {X_tr_raw.max()}]")

In [ ]:
# Pandas EDA
flat = X_tr_raw.reshape(50_000, -1)          # 50000 × 3072
df_eda = pd.DataFrame({
    "label"   : y_tr_raw,
    "class"   : [CLASS_NAMES[i] for i in y_tr_raw],
    "mean_R"  : flat[:, 0:1024].mean(1),
    "mean_G"  : flat[:, 1024:2048].mean(1),
    "mean_B"  : flat[:, 2048:].mean(1),
    "brightness": flat.mean(1),
})
print("\n  Pandas head (first 5 rows):")
print(df_eda.head())
print("\n  Class distribution (training):")
print(df_eda["class"].value_counts().to_string())
print("\n  Channel statistics:")
print(df_eda[["mean_R","mean_G","mean_B","brightness"]].describe().round(2))


In [ ]:
# sample grid + class distribution
fig = plt.figure(figsize=(22, 11))
_style(fig)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Sample grid (3 × 10)
ax_grid = fig.add_subplot(gs[0, :2])
ax_grid.axis("off")
ax_grid.set_facecolor(BG_DARK)
ax_grid.set_title("Sample Images — 3 per Class", color="white", fontsize=14, pad=8)
for ci, cname in enumerate(CLASS_NAMES):
    idxs = np.where(y_tr_raw == ci)[0][:3]
    for j, idx in enumerate(idxs):
        ax_s = fig.add_axes([0.03 + ci*0.087, 0.58 - j*0.145, 0.075, 0.13])
        ax_s.imshow(X_tr_raw[idx])
        if j == 0:
            ax_s.set_title(cname, color="white", fontsize=7)
        ax_s.axis("off")
        
# Class distribution
ax_dist = fig.add_subplot(gs[0, 2])
_style(fig, ax_dist)
counts = df_eda["class"].value_counts().loc[CLASS_NAMES]
clrs   = sns.color_palette("plasma", 10)
ax_dist.barh(CLASS_NAMES, counts.values, color=clrs, edgecolor="white", lw=0.4)
ax_dist.set_title("Class Distribution", color="white", fontsize=13)
ax_dist.set_xlabel("Count", color="#aaaaaa")
ax_dist.tick_params(colors="white")
for v, bar in zip(counts.values, ax_dist.patches):
    ax_dist.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2,
                 f"{v:,}", va="center", color="white", fontsize=8)
